<a href="https://colab.research.google.com/github/bilalsherifdeen1-ux/arrowstack-ames-price-prediction/blob/main/Arrowstack4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import fetch_openml

# Task 1
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto").frame

# Task 2 (fallback if Telco isn't on hand)
churn = fetch_openml(name="churn", as_frame=True, parser="auto").frame

# Task 3: UCI SMS Spam (tab-separated: label, text)
import pandas as pd, io, zipfile, urllib.request
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
sms = pd.read_csv(z.open("SMSSpamCollection"), sep="\t", names=["label", "text"])

/usr/local/lib/python3.13/dist-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name churn exist. Versions may be fundamentally different, returning version 1. Available versions:
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=40701
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=45076

  warn(warning_msg)


In [ ]:
from sklearn.datasets import fetch_openml

# Why: data_id removes ambiguity; the name alone can resolve to different versions
churn = fetch_openml(data_id=40701, as_frame=True, parser="auto").frame
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto").frame

print(ames.shape, churn.shape)
print(churn.dtypes.value_counts())
print(churn.iloc[:, -1].value_counts(normalize=True))  # check the label column

(1460, 81) (5000, 21)
int64       8
float64     8
category    3
category    1
category    1
Name: count, dtype: int64
class
0    0.8586
1    0.1414
Name: proportion, dtype: float64


In [ ]:
print(ames["SalePrice"].describe())
print("skew:", ames["SalePrice"].skew())
print(ames.isna().mean().sort_values().tail(10))

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64
skew: 1.8828757597682129
GarageFinish    0.055479
GarageType      0.055479
GarageCond      0.055479
GarageQual      0.055479
LotFrontage     0.177397
FireplaceQu     0.472603
Fence           0.807534
Alley           0.937671
MiscFeature     0.963014
PoolQC          0.995205
dtype: float64


In [ ]:
print(churn["phone_number"].nunique(), len(churn))
print(churn.groupby("class")["number_customer_service_calls"].apply(lambda s: s.astype(int).mean()))
print(churn.select_dtypes("number").corr().loc["total_day_minutes", "total_day_charge"])

5000 5000
class
0    1.457722
1    2.254597
Name: number_customer_service_calls, dtype: float64
0.9999999509831975


/tmp/ipykernel_1052/3113393144.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(churn.groupby("class")["number_customer_service_calls"].apply(lambda s: s.astype(int).mean()))


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
SEED = 42
num_pipe = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())])
cat_pipe = Pipeline([("i", SimpleImputer(strategy="constant", fill_value="missing")), ("o", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("num", num_pipe, make_column_selector(dtype_include="number")), ("cat", cat_pipe, make_column_selector(dtype_exclude="number"))])

In [ ]:
a = ames.drop(columns=["Id"], errors="ignore")
Xa = a.drop(columns="SalePrice")
ya = a["SalePrice"]
Xa = Xa.astype({k: "object" for k in Xa.select_dtypes(exclude="number").columns})
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.2, random_state=SEED)
cva = KFold(5, shuffle=True, random_state=SEED)
m1 = Pipeline([("pre", pre), ("m", TransformedTargetRegressor(DummyRegressor(strategy="median"), func=np.log1p, inverse_func=np.expm1))])
m2 = Pipeline([("pre", pre), ("m", TransformedTargetRegressor(Ridge(alpha=10), func=np.log1p, inverse_func=np.expm1))])
print("Ames median MAE:", round(-cross_val_score(m1, Xa_tr, ya_tr, cv=cva, scoring="neg_mean_absolute_error").mean()))
print("Ames ridge MAE:", round(-cross_val_score(m2, Xa_tr, ya_tr, cv=cva, scoring="neg_mean_absolute_error").mean()))

Ames median MAE: 54586
Ames ridge MAE: 17313


In [ ]:
c = churn.drop(columns=["phone_number"])
yc = (c["class"].astype(int) == 1).astype(int)
Xc = c.drop(columns="class")
Xc["state"] = Xc["state"].astype(int).astype(str)
Xc["number_customer_service_calls"] = Xc["number_customer_service_calls"].astype(int)
Xc = Xc.astype({k: "object" for k in Xc.select_dtypes(exclude="number").columns})
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=SEED)
cvc = StratifiedKFold(5, shuffle=True, random_state=SEED)
d = Pipeline([("pre", pre), ("m", DummyClassifier(strategy="prior"))])
l = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))])
print("Churn dummy PR-AUC:", round(cross_val_score(d, Xc_tr, yc_tr, cv=cvc, scoring="average_precision").mean(), 3))
print("Churn logreg PR-AUC:", round(cross_val_score(l, Xc_tr, yc_tr, cv=cvc, scoring="average_precision").mean(), 3))

Churn dummy PR-AUC: 0.142
Churn logreg PR-AUC: 0.456


In [ ]:
fe_ames = lambda X: X.assign(HouseAge=X["YrSold"] - X["YearBuilt"])
s0 = -cross_val_score(m2, Xa_tr, ya_tr, cv=cva, scoring="neg_mean_absolute_error")
s1 = -cross_val_score(m2, fe_ames(Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")
print("base MAE:", round(s0.mean()), "+/-", round(s0.std()))
print("fe   MAE:", round(s1.mean()), "+/-", round(s1.std()))

base MAE: 17313 +/- 1541
fe   MAE: 17313 +/- 1543


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
rf = Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1))])
xgb = Pipeline([("pre", pre), ("m", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, scale_pos_weight=(yc_tr==0).sum()/(yc_tr==1).sum(), random_state=SEED, eval_metric="logloss"))])
s_rf = cross_val_score(rf, Xc_tr, yc_tr, cv=cvc, scoring="average_precision")
s_xgb = cross_val_score(xgb, Xc_tr, yc_tr, cv=cvc, scoring="average_precision")
print("rf  PR-AUC:", round(s_rf.mean(), 3), "+/-", round(s_rf.std(), 3))
print("xgb PR-AUC:", round(s_xgb.mean(), 3), "+/-", round(s_xgb.std(), 3))

rf  PR-AUC: 0.863 +/- 0.027
xgb PR-AUC: 0.863 +/- 0.024


In [ ]:
drop_chg = ["total_day_charge","total_eve_charge","total_night_charge","total_intl_charge"]
s_a = cross_val_score(xgb, Xc_tr.drop(columns=drop_chg), yc_tr, cv=cvc, scoring="average_precision")
s_b = cross_val_score(xgb, Xc_tr.drop(columns=["state"]), yc_tr, cv=cvc, scoring="average_precision")
print("no charge cols:", round(s_a.mean(), 3), "+/-", round(s_a.std(), 3))
print("no state     :", round(s_b.mean(), 3), "+/-", round(s_b.std(), 3))

no charge cols: 0.863 +/- 0.023
no state     : 0.871 +/- 0.023


In [ ]:
d = s_b - s_xgb
print(d.round(3), "mean diff:", round(d.mean(), 4))

[0.005 0.013 0.004 0.01  0.011] mean diff: 0.0086


In [ ]:
drop_all = drop_chg + ["state"]
s_c = cross_val_score(xgb, Xc_tr.drop(columns=drop_all), yc_tr, cv=cvc, scoring="average_precision")
print("simplified:", round(s_c.mean(), 3), "+/-", round(s_c.std(), 3))

simplified: 0.868 +/- 0.021


In [ ]:
print([c for c in Xa.columns if "Area" in c or "SF" in c or "Qual" in c])
feats = {}
feats["log_area"] = lambda X: X.assign(LogArea=np.log1p(X["GrLivArea"]))
# feats["your_feature"] = lambda X: X.assign(...)
res = {k: round((-cross_val_score(m2, f(Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")).mean()) for k, f in feats.items()}
print("baseline: 17313", res)

['LotArea', 'OverallQual', 'MasVnrArea', 'ExterQual', 'BsmtQual', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'KitchenQual', 'GarageArea', 'GarageQual', 'WoodDeckSF', 'OpenPorchSF', 'PoolArea']
baseline: 17313 {'log_area': 16224}


In [ ]:
s_log = -cross_val_score(m2, fe_ames(Xa_tr).assign(LogArea=np.log1p(Xa_tr["GrLivArea"])), ya_tr, cv=cva, scoring="neg_mean_absolute_error")
print((s0 - s_log).round(0), "mean gain:", round((s0 - s_log).mean()))

[  71. 1249. 2195. 1204.  717.] mean gain: 1087


In [ ]:
print(Xa[["OverallQual", "ExterQual", "KitchenQual"]].dtypes)
print(Xa["KitchenQual"].value_counts(dropna=False))

OverallQual     int64
ExterQual      object
KitchenQual    object
dtype: object
KitchenQual
TA    735
Gd    586
Ex    100
Fa     39
Name: count, dtype: int64


In [ ]:
q = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1}
print(Xa["ExterQual"].value_counts(dropna=False))
feats["qual_ord"] = lambda X: X.assign(KitchenN=X["KitchenQual"].map(q), ExterN=X["ExterQual"].map(q))
feats["log+qual"] = lambda X: feats["qual_ord"](feats["log_area"](X))
res = {k: round((-cross_val_score(m2, f(Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")).mean()) for k, f in feats.items()}
print("baseline: 17313", res)

ExterQual
TA    906
Gd    488
Ex     52
Fa     14
Name: count, dtype: int64
baseline: 17313 {'log_area': 16224, 'qual_ord': 17291, 'log+qual': 16214}


In [ ]:
feats["qual_x_area"] = lambda X: X.assign(LogArea=np.log1p(X["GrLivArea"]), QxA=X["OverallQual"] * np.log1p(X["GrLivArea"]))
s_int = -cross_val_score(m2, feats["qual_x_area"](Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")
s_log = -cross_val_score(m2, feats["log_area"](Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")
print("interaction MAE:", round(s_int.mean()), "| fold gains vs LogArea:", (s_log - s_int).round(0), "| mean:", round((s_log - s_int).mean()))

interaction MAE: 16218 | fold gains vs LogArea: [ 59.  -7.  41. -15. -45.] | mean: 7


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
rf_a = Pipeline([("pre", pre), ("m", TransformedTargetRegressor(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), func=np.log1p, inverse_func=np.expm1))])
gb_a = Pipeline([("pre", pre), ("m", TransformedTargetRegressor(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=SEED), func=np.log1p, inverse_func=np.expm1))])
s_rf = -cross_val_score(rf_a, Xa_tr, ya_tr, cv=cva, scoring="neg_mean_absolute_error")
s_gb = -cross_val_score(gb_a, Xa_tr, ya_tr, cv=cva, scoring="neg_mean_absolute_error")
print("RF MAE:", round(s_rf.mean()), "+/-", round(s_rf.std()), "| GB MAE:", round(s_gb.mean()), "+/-", round(s_gb.std()))
print("GB vs Ridge+LogArea fold gains:", (s_log - s_gb).round(0))

RF MAE: 18363 +/- 841 | GB MAE: 15916 +/- 1242
GB vs Ridge+LogArea fold gains: [ 1002. -1166.   429.   311.   966.]


In [ ]:
s_gb2 = -cross_val_score(gb_a, feats["log_area"](Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")
print("GB + LogArea MAE:", round(s_gb2.mean()), "| diff vs GB:", (s_gb - s_gb2).round(0))

GB + LogArea MAE: 15839 | diff vs GB: [  0. 166. 172. -53.  98.]


In [ ]:
mk = lambda a_: Pipeline([("pre", pre), ("m", TransformedTargetRegressor(Ridge(alpha=a_), func=np.log1p, inverse_func=np.expm1))])
print({a_: round((-cross_val_score(mk(a_), feats["log_area"](Xa_tr), ya_tr, cv=cva, scoring="neg_mean_absolute_error")).mean()) for a_ in [1, 3, 10, 30, 100]})

{1: 16604, 3: 16340, 10: 16224, 30: 16562, 100: 17131}


In [ ]:
from sklearn.model_selection import cross_val_predict
pr = cross_val_predict(mk(10), feats["log_area"](Xa_tr), ya_tr, cv=cva)
pg = cross_val_predict(gb_a, Xa_tr, ya_tr, cv=cva)
band = pd.qcut(ya_tr, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
print(pd.DataFrame({"ridge_MAE": (ya_tr - pr).abs(), "gb_MAE": (ya_tr - pg).abs(), "ridge_pct": (ya_tr - pr).abs() / ya_tr * 100, "band": band}).groupby("band", observed=True).mean().round(1))
print("worst 5 ridge errors:", (ya_tr - pr).abs().sort_values(ascending=False).head(5).round(0).to_dict())

         ridge_MAE   gb_MAE  ridge_pct
band                                  
Q1 low     11360.2  12266.2       11.6
Q2         14219.5  10853.7        9.5
Q3         14255.7  12909.0        7.7
Q4 high    25076.9  27665.3        8.1
worst 5 ridge errors: {1298: 913513.0, 523: 365847.0, 1182: 265813.0, 803: 153466.0, 185: 149089.0}


In [ ]:
big = Xa_tr["GrLivArea"] > 4000
print("n flagged:", int(big.sum()))
print(pd.concat([Xa_tr.loc[big, ["GrLivArea", "OverallQual", "SaleCondition"]], ya_tr[big]], axis=1))
print("ridge MAE all / excl:", round((ya_tr - pr).abs().mean()), round((ya_tr - pr).abs()[~big].mean()))
print("gb    MAE all / excl:", round((ya_tr - pg).abs().mean()), round((ya_tr - pg).abs()[~big].mean()))
print("median AE ridge / gb:", round((ya_tr - pr).abs().median()), round((ya_tr - pg).abs().median()))

n flagged: 3
      GrLivArea  OverallQual SaleCondition  SalePrice
1182       4476           10       Abnorml     745000
523        4676           10       Partial     184750
1298       5642           10       Partial     160000
ridge MAE all / excl: 16226 14941
gb    MAE all / excl: 15918 15236
median AE ridge / gb: 10509 10017


In [ ]:
mk_gb = lambda: Pipeline([("pre", pre), ("m", TransformedTargetRegressor(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=SEED), func=np.log1p, inverse_func=np.expm1))])
Xl = feats["log_area"](Xa_tr)
pr2 = np.zeros(len(ya_tr)); pg2 = np.zeros(len(ya_tr))
for tr, te in cva.split(Xa_tr): tr = tr[~big.values[tr]]; pr2[te] = mk(10).fit(Xl.iloc[tr], ya_tr.iloc[tr]).predict(Xl.iloc[te]); pg2[te] = mk_gb().fit(Xa_tr.iloc[tr], ya_tr.iloc[tr]).predict(Xa_tr.iloc[te])
ae = lambda p: pd.Series(np.abs(ya_tr.values - p))
print("ridge all / 3 flagged only / median:", round(ae(pr2).mean()), round(ae(pr2)[big.values].mean()), round(ae(pr2).median()))
print("gb    all / 3 flagged only / median:", round(ae(pg2).mean()), round(ae(pg2)[big.values].mean()), round(ae(pg2).median()))

ridge all / 3 flagged only / median: 15729 907739 9342
gb    all / 3 flagged only / median: 15894 374784 9967


In [ ]:
Xl_te = feats["log_area"](Xa_te)
keep = ~big.values
fin = mk(10).fit(Xl.iloc[np.where(keep)[0]], ya_tr.iloc[np.where(keep)[0]])
pt = fin.predict(Xl_te)
base = Pipeline([("pre", pre), ("m", TransformedTargetRegressor(DummyRegressor(strategy="median"), func=np.log1p, inverse_func=np.expm1))]).fit(Xa_tr, ya_tr)
ae_t = lambda p: np.abs(ya_te.values - p)
print("TEST ridge MAE / median:", round(ae_t(pt).mean()), round(np.median(ae_t(pt))))
print("TEST baseline MAE / median:", round(ae_t(base.predict(Xa_te)).mean()), round(np.median(ae_t(base.predict(Xa_te)))))
print("test houses >4000 sqft:", int((Xa_te["GrLivArea"] > 4000).sum()))

TEST ridge MAE / median: 14313 10100
TEST baseline MAE / median: 59568 40000
test houses >4000 sqft: 1


In [ ]:
from sklearn.model_selection import cross_val_predict
sc_rf = cross_val_score(rf, Xc_tr, yc_tr, cv=cvc, scoring="average_precision")
sc_xgb = cross_val_score(xgb, Xc_tr, yc_tr, cv=cvc, scoring="average_precision")
dd = sc_xgb - sc_rf
print("XGB - RF fold diffs:", dd.round(3), "mean:", round(dd.mean(), 4))

XGB - RF fold diffs: [ 0.004  0.004  0.002 -0.018  0.004] mean: -0.0007


In [ ]:
Xc2 = Xc_tr.drop(columns=["state"])
yv = yc_tr.values
topk = lambda p: np.argsort(-p)[: int(0.10 * len(p))]
pp = {n: cross_val_predict(m, Xc2, yc_tr, cv=cvc, method="predict_proba")[:, 1] for n, m in [("rf", rf), ("xgb", xgb)]}
print({n: (round(yv[topk(p)].mean(), 3), round(yv[topk(p)].sum() / yv.sum(), 3)) for n, p in pp.items()}, "(precision@10%, recall@10%)")

{'rf': (np.float64(0.96), np.float64(0.678)), 'xgb': (np.float64(0.96), np.float64(0.678))} (precision@10%, recall@10%)


In [ ]:
pr_x = pp["xgb"]
print({k: (round(yv[np.argsort(-pr_x)[: int(k * len(pr_x))]].mean(), 3), round(yv[np.argsort(-pr_x)[: int(k * len(pr_x))]].sum() / yv.sum(), 3)) for k in [0.05, 0.10, 0.15, 0.20, 0.30]}, "(precision, recall) by share contacted")
print("top-400 overlap RF vs XGB:", len(set(topk(pp["rf"])) & set(topk(pp["xgb"]))))

{0.05: (np.float64(1.0), np.float64(0.353)), 0.1: (np.float64(0.96), np.float64(0.678)), 0.15: (np.float64(0.782), np.float64(0.829)), 0.2: (np.float64(0.605), np.float64(0.855)), 0.3: (np.float64(0.416), np.float64(0.882))} (precision, recall) by share contacted
top-400 overlap RF vs XGB: 356


In [ ]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score
dm = Pipeline([("pre", pre), ("m", DummyClassifier(strategy="prior"))])
lr = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))])
Xtr2, Xte2, yt = Xc_tr.drop(columns=["state"]), Xc_te.drop(columns=["state"]), yc_te.values
k = int(0.10 * len(yt))
sc = {n: clone(m).fit(Xtr2, yc_tr).predict_proba(Xte2)[:, 1] for n, m in [("dummy", dm), ("logreg", lr), ("rf", rf), ("xgb", xgb)]}
print({n: round(average_precision_score(yt, p), 3) for n, p in sc.items()}, "TEST PR-AUC")
print({n: (round(yt[np.argsort(-p)[:k]].mean(), 3), round(yt[np.argsort(-p)[:k]].sum() / yt.sum(), 3)) for n, p in sc.items() if n != "dummy"}, "(precision@10%, recall@10%) TEST")
print("test churn rate:", round(yt.mean(), 3), "| n test:", len(yt))

{'dummy': np.float64(0.141), 'logreg': np.float64(0.468), 'rf': np.float64(0.851), 'xgb': np.float64(0.855)} TEST PR-AUC
{'logreg': (np.float64(0.47), np.float64(0.333)), 'rf': (np.float64(0.92), np.float64(0.652)), 'xgb': (np.float64(0.96), np.float64(0.681))} (precision@10%, recall@10%) TEST
test churn rate: 0.141 | n test: 1000


In [ ]:
import sklearn, xgboost, platform
print("python", platform.python_version(), "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

python 3.13.16 | sklearn 1.6.1 | xgboost 3.4.1 | pandas 2.2.3 | numpy 2.1.3


In [ ]:
b1 = fetch_openml(name="house_prices", as_frame=True, parser="auto")
b2 = fetch_openml(data_id=40701, as_frame=True, parser="auto")
print({k: (b.details.get("id"), b.details.get("version"), b.details.get("licence"), b.details.get("upload_date")) for k, b in [("ames", b1), ("churn", b2)]})

{'ames': ('42165', '1', 'NA', '2019-10-04T12:57:32'), 'churn': ('40701', '1', 'public', '2017-04-06T12:37:40')}


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import cross_val_predict
print("missing:", [n for n in ["Xc_tr", "yc_tr", "rf", "xgb", "cvc"] if n not in globals()])

missing: []


In [ ]:
Xc2 = Xc_tr.drop(columns=["state"])
yv = yc_tr.values
pp = {n: cross_val_predict(m, Xc2, yc_tr, cv=cvc, method="predict_proba")[:, 1] for n, m in [("rf", rf), ("xgb", xgb)]}

In [ ]:
cut = np.argsort(-pp["xgb"])[: int(0.10 * len(yv))]
caught = np.zeros(len(yv), bool)
caught[cut] = True
missed = (yv == 1) & ~caught
hit = (yv == 1) & caught
cols = ["number_customer_service_calls", "total_day_minutes", "total_eve_minutes", "total_intl_calls", "number_vmail_messages"]
print(pd.DataFrame({"caught": Xc2.loc[hit, cols].astype(float).mean(), "missed": Xc2.loc[missed, cols].astype(float).mean(), "non-churn": Xc2.loc[yv == 0, cols].astype(float).mean()}).round(2))
print("caught:", int(hit.sum()), "missed:", int(missed.sum()))

                               caught  missed  non-churn
number_customer_service_calls    2.27    2.22       1.46
total_day_minutes              210.75  204.09     175.95
total_eve_minutes              213.25  206.34     199.36
total_intl_calls                 4.08    4.44       4.51
number_vmail_messages            3.85    5.42       8.30
caught: 384 missed: 182


In [ ]:
order = np.argsort(-pp["xgb"])
rank = np.empty(len(order), int)
rank[order] = np.arange(1, len(order) + 1)
print("missed ranks 401-600:", int(((rank[missed] > 400) & (rank[missed] <= 600)).sum()), "| beyond 600:", int((rank[missed] > 600).sum()))

In [ ]:
deep = missed & (rank > 600)
near = missed & (rank > 400) & (rank <= 600)
print(pd.DataFrame({"near": Xc2.loc[near, cols].astype(float).mean(), "deep": Xc2.loc[deep, cols].astype(float).mean(), "non-churn": Xc2.loc[yv == 0, cols].astype(float).mean()}).round(2))

                                 near    deep  non-churn
number_customer_service_calls    2.79    1.72       1.46
total_day_minutes              222.29  188.14     175.95
total_eve_minutes              212.41  201.02     199.36
total_intl_calls                 4.66    4.25       4.51
number_vmail_messages            3.44    7.16       8.30
